# KALPA case study


## 1. Setup and data loading

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path(".")  # folder containing the CSV files
TABLES = [
    "channels", "customers", "marketing_spend", "members", "member_events",
    "orders", "order_costs", "periods", "renewal_opportunities",
]
DIGITAL = ["APP", "WEB"]

# Every file is read exactly once; later cells reuse these frames.
data = {name: pd.read_csv(DATA_DIR / f"{name}.csv") for name in TABLES}
channels, customers, mkt, members, events, orders, order_costs, periods, renewals = (
    data[t] for t in TABLES
)

for name, frame in data.items():
    print(f"\n--- {name}.csv ---")
    print("Shape:", frame.shape)
    print("Columns:", frame.columns.tolist())
    print(frame.head(2).to_string())


--- channels.csv ---
Shape: (3, 2)
Columns: ['channel_id', 'channel_name']
  channel_id channel_name
0        APP   Mobile app
1        WEB      Website

--- customers.csv ---
Shape: (18000, 5)
Columns: ['customer_id', 'segment', 'city', 'region', 'signup_date']
  customer_id   segment       city region signup_date
0     C000001  Business      Delhi  North  2023-02-10
1     C000002  Business  Hyderabad  South  2023-10-25

--- marketing_spend.csv ---
Shape: (144, 6)
Columns: ['spend_id', 'period_id', 'month_start', 'channel_id', 'purpose', 'amount']
  spend_id  period_id month_start channel_id      purpose  amount
0   M00001  FY2024_25  2024-04-01        APP  Acquisition  960718
1   M00002  FY2024_25  2024-04-01        APP    Retention  535845

--- members.csv ---
Shape: (8580, 8)
Columns: ['member_id', 'join_date', 'plan_code', 'annual_fee', 'acquisition_channel', 'city', 'status_2025_10_09', 'status_2026_10_09']
  member_id   join_date      plan_code  annual_fee acquisition_channel  

In [2]:
# Key integrity: the joins below assume these keys are unique.
for name, key in [("orders", "order_id"), ("order_costs", "order_id"),
                  ("customers", "customer_id"), ("members", "member_id"),
                  ("renewal_opportunities", "renewal_id")]:
    assert data[name][key].is_unique, f"{name}.{key} contains duplicates"

# Orphan checks (printed rather than asserted so one bad row does not stop the notebook).
print("Orders with channel not in channels.csv:", (~orders["channel_id"].isin(channels["channel_id"])).sum())
print("Orders with no cost row:", (~orders["order_id"].isin(order_costs["order_id"])).sum())
print("Orders with no customer row:", (~orders["customer_id"].isin(customers["customer_id"])).sum())
print("Renewals with no member row:", (~renewals["member_id"].isin(members["member_id"])).sum())
print("customer_id / member_id overlap:", len(set(customers["customer_id"]) & set(members["member_id"])))

# Quick categorical profile
print("\nCustomer segments:\n", customers["segment"].value_counts())
print("\nOrder statuses:\n", orders["status"].value_counts())
print("\nPeriods:\n", periods.to_string())

Orders with channel not in channels.csv: 0
Orders with no cost row: 0
Orders with no customer row: 0
Renewals with no member row: 0
customer_id / member_id overlap: 0

Customer segments:
 segment
Retail-Core    9945
Retail-Plus    5070
Student        2092
Business        893
Name: count, dtype: int64

Order statuses:
 status
COMPLETED    102000
CANCELLED      4400
Name: count, dtype: int64

Periods:
    period_id period_label  start_date    end_date      role
0  FY2024_25   FY 2024-25  2024-04-01  2025-03-31  previous
1  FY2025_26   FY 2025-26  2025-04-01  2026-03-31   current


## 2. Order financials
Revenue and profit metrics are calculated on **COMPLETED** orders only. 

In [3]:
df = (
    orders
    .merge(order_costs, on="order_id", how="left", validate="1:1")
    .merge(customers, on="customer_id", how="left", validate="m:1")
)
assert len(df) == len(orders), "a merge changed the row count"

df["net_sales"] = df["gross_amount"] - df["discount_amount"] - df["refund_amount"]
df["gross_profit"] = df["net_sales"] - df["product_cost"]
df["order_contribution"] = df["gross_profit"] - df["fulfilment_cost"] - df["payment_cost"]

completed = df[df["status"] == "COMPLETED"]
print(df.groupby(["period_id", "status"])["order_id"].count())

period_id  status   
FY2024_25  CANCELLED     2000
           COMPLETED    50000
FY2025_26  CANCELLED     2400
           COMPLETED    52000
Name: order_id, dtype: int64


In [4]:
def pnl(frame, by):
    """Aggregate order-level P&L and ratios for any grouping."""
    out = frame.groupby(by).agg(
        orders=("order_id", "count"),
        customers=("customer_id", "nunique"),
        gross_sales=("gross_amount", "sum"),
        discounts=("discount_amount", "sum"),
        refunds=("refund_amount", "sum"),
        net_sales=("net_sales", "sum"),
        cogs=("product_cost", "sum"),
        gross_profit=("gross_profit", "sum"),
        fulfilment=("fulfilment_cost", "sum"),
        payment_fee=("payment_cost", "sum"),
        order_contribution=("order_contribution", "sum"),
    ).reset_index()
    out["aov"] = out["net_sales"] / out["orders"]
    out["sales_per_cust"] = out["net_sales"] / out["customers"]
    out["discount_pct"] = out["discounts"] / out["gross_sales"] * 100
    out["gross_margin_pct"] = out["gross_profit"] / out["net_sales"] * 100
    out["order_contrib_pct"] = out["order_contribution"] / out["net_sales"] * 100
    return out


def with_marketing(summary, by):
    """Attach marketing spend (left join, so groups with no spend are kept) and net contribution."""
    spend = mkt.groupby(by, as_index=False)["amount"].sum().rename(columns={"amount": "marketing_spend"})
    out = summary.merge(spend, on=by, how="left")
    out["marketing_spend"] = out["marketing_spend"].fillna(0)
    out["net_contribution"] = out["order_contribution"] - out["marketing_spend"]
    out["net_contrib_pct"] = out["net_contribution"] / out["net_sales"] * 100
    return out


period_summary = with_marketing(pnl(completed, ["period_id"]), ["period_id"])
print("Financial summary (completed orders):")
print(period_summary.set_index("period_id").T)

Financial summary (completed orders):
period_id              FY2024_25     FY2025_26
orders              5.000000e+04  5.200000e+04
customers           1.000000e+04  1.300000e+04
gross_sales         1.068970e+09  1.143714e+09
discounts           6.460143e+07  9.915585e+07
refunds             4.368794e+06  4.558595e+06
net_sales           1.000000e+09  1.040000e+09
cogs                6.800000e+08  7.160000e+08
gross_profit        3.200000e+08  3.240000e+08
fulfilment          4.000000e+07  5.720000e+07
payment_fee         1.000000e+07  1.560000e+07
order_contribution  2.700000e+08  2.512000e+08
aov                 2.000000e+04  2.000000e+04
sales_per_cust      1.000000e+05  8.000000e+04
discount_pct        6.043333e+00  8.669633e+00
gross_margin_pct    3.200000e+01  3.115385e+01
order_contrib_pct   2.700000e+01  2.415385e+01
marketing_spend     7.000000e+07  9.520000e+07
net_contribution    2.000000e+08  1.560000e+08
net_contrib_pct     2.000000e+01  1.500000e+01


In [5]:
channel_summary = with_marketing(pnl(completed, ["period_id", "channel_id"]), ["period_id", "channel_id"])
print(channel_summary[["period_id", "channel_id", "orders", "net_sales", "gross_profit", "marketing_spend",
                       "order_contribution", "net_contribution", "discount_pct", "gross_margin_pct",
                       "net_contrib_pct"]].to_string())

segment_summary = pnl(completed, ["period_id", "segment"])
print("\nSegment performance:")
print(segment_summary[["period_id", "segment", "orders", "customers", "net_sales", "gross_profit",
                       "aov", "sales_per_cust"]].to_string())

   period_id channel_id  orders  net_sales  gross_profit  marketing_spend  order_contribution  net_contribution  discount_pct  gross_margin_pct  net_contrib_pct
0  FY2024_25        APP   20796  275359673      88729651         23578320            71645649          48067329      6.033192         32.223183        17.456198
1  FY2024_25      STORE   13999  402840912     127938322         24593278           110735051          86141773      6.033549         31.759019        21.383571
2  FY2024_25        WEB   15205  321799415     103332027         21828402            87619300          65790898      6.064268         32.110694        20.444692
3  FY2025_26        APP   28167  330181667     102953818         32402894            72397723          39994829      8.614751         31.180961        12.112977
4  FY2025_26      STORE   10202  392754610     122826041         31000056           101391222          70391166      8.697060         31.272972        17.922429
5  FY2025_26        WEB   13631  3

In [6]:
# Monthly trend: look for step changes in discounting and unit costs
monthly = (
    completed.assign(order_month=pd.to_datetime(completed["order_date"]).dt.to_period("M"))
    .groupby("order_month")
    .agg(orders=("order_id", "count"), gross=("gross_amount", "sum"), discount=("discount_amount", "sum"),
         product_cost=("product_cost", "sum"), fulfilment_cost=("fulfilment_cost", "sum"),
         payment_cost=("payment_cost", "sum"))
)
monthly["disc_pct"] = monthly["discount"] / monthly["gross"] * 100
monthly["cogs_pct"] = monthly["product_cost"] / (monthly["gross"] - monthly["discount"]) * 100
monthly["fulfil_per_order"] = monthly["fulfilment_cost"] / monthly["orders"]
monthly["payment_per_order"] = monthly["payment_cost"] / monthly["orders"]
print(monthly[["orders", "disc_pct", "fulfil_per_order", "payment_per_order"]].to_string())

             orders  disc_pct  fulfil_per_order  payment_per_order
order_month                                                       
2024-04        3746  6.099828        803.389749         201.002136
2024-05        3652  6.079717        796.597481         194.653067
2024-06        3759  6.085631        788.387869         193.018356
2024-07        3887  5.918631        781.067919         194.323128
2024-08        3738  6.182960        807.512039         202.525147
2024-09        4242  5.898869        785.017916         192.346770
2024-10        5743  6.077842        797.892391         199.546578
2024-11        5273  5.974105        809.627726         205.005689
2024-12        4882  6.129092        808.402089         206.778165
2025-01        3863  6.104253        799.435413         198.213306
2025-02        3738  5.917459        805.692349         197.337346
2025-03        3477  6.044742        815.433420         213.540984
2025-04        4238  8.585948       1069.068664         284.03

## 3. Omnichannel vs single-channel customers

In [7]:
def omni_profile(frame):
    """One row per customer per period, labelled Omnichannel / Pure Offline / Pure Digital."""
    cust = (
        frame.assign(is_store=frame["channel_id"].eq("STORE"), is_digital=frame["channel_id"].isin(DIGITAL))
        .groupby(["period_id", "customer_id"])
        .agg(has_store=("is_store", "any"), has_digital=("is_digital", "any"),
             net_sales=("net_sales", "sum"), orders=("order_id", "count"))
        .reset_index()
    )
    cust["omni_type"] = np.select(
        [cust["has_store"] & cust["has_digital"], cust["has_store"]],
        ["Omnichannel (Store + Digital)", "Pure Offline (Store Only)"],
        default="Pure Digital (App/Web Only)",
    )
    cust["aov"] = cust["net_sales"] / cust["orders"]
    return cust


def omni_report(frame, title):
    cust = omni_profile(frame)
    print(f"\n=== {title} ===")
    print(cust.groupby(["period_id", "omni_type"]).agg(
        customers=("customer_id", "count"), orders=("orders", "mean"), aov=("aov", "mean"),
        avg_spend=("net_sales", "mean"), total_sales=("net_sales", "sum")))
    # Spend multiple of omnichannel vs each single-channel group (computed, not hard-coded)
    spend = cust.pivot_table(index="period_id", columns="omni_type", values="net_sales", aggfunc="mean")
    omni = "Omnichannel (Store + Digital)"
    print("\nOmnichannel avg spend as a multiple of:")
    print(spend.drop(columns=omni).rdiv(spend[omni], axis=0))


omni_report(completed, "All segments")
omni_report(completed[completed["segment"] != "Business"], "Excluding Business segment")


=== All segments ===
                                         customers    orders           aov  \
period_id omni_type                                                          
FY2024_25 Omnichannel (Store + Digital)       5881  6.794933  18234.683012   
          Pure Digital (App/Web Only)         3575  2.591608  11141.958056   
          Pure Offline (Store Only)            544  1.422794  13413.042678   
FY2025_26 Omnichannel (Store + Digital)       5051  6.169669  21459.294780   
          Pure Digital (App/Web Only)         7379  2.721100  10095.332780   
          Pure Offline (Store Only)            570  1.329825  19355.818843   

                                             avg_spend  total_sales  
period_id omni_type                                                  
FY2024_25 Omnichannel (Store + Digital)  150017.701581    882254103  
          Pure Digital (App/Web Only)     29178.869371    104314458  
          Pure Offline (Store Only)       24690.145221     13431439  
FY2

## 4. Marketing spend and membership profile

In [8]:
print("Marketing spend:")
print(mkt.groupby(["period_id", "channel_id", "purpose"])["amount"].sum())

members.info()
for col in ["status_2025_10_09", "status_2026_10_09", "plan_code", "acquisition_channel"]:
    print(f"\n{col}:\n", members[col].value_counts())

print("\nMember events: type x outcome")
print(events.groupby(["event_type", "outcome"])["event_id"].count())

Marketing spend:
period_id  channel_id  purpose    
FY2024_25  APP         Acquisition    15554325
                       Retention       8023995
           STORE       Acquisition    14045373
                       Retention      10547905
           WEB         Acquisition    13197679
                       Retention       8630723
FY2025_26  APP         Acquisition    19889541
                       Retention      12513353
           STORE       Acquisition    18275668
                       Retention      12724388
           WEB         Acquisition    18503855
                       Retention      13293195
Name: amount, dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8580 entries, 0 to 8579
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   member_id            8580 non-null   object
 1   join_date            8580 non-null   object
 2   plan_code            8580 non-null   object
 3   a

## 5. Renewals
A renewal counts as successful if it has at least one successful `PAYMENT_ATTEMPT`. Renewals whose deadline is **after the last event in the data are still open**, so they are not failures. 

In [9]:
as_of = pd.to_datetime(events["event_time"]).max()   # data cut-off proxy
paid_ids = set(events.loc[(events["event_type"] == "PAYMENT_ATTEMPT") & (events["outcome"] == "SUCCESS"),
                          "renewal_id"])

ren = renewals.merge(members, on="member_id", how="left", validate="m:1")
ren["expiry_year"] = pd.to_datetime(ren["expiry_date"]).dt.year
ren["is_success"] = ren["renewal_id"].isin(paid_ids)
ren["is_open"] = pd.to_datetime(ren["renewal_deadline"]) > as_of
ren_closed = ren[~ren["is_open"]]

print("Data cut-off (max event time):", as_of)
print(f"Total renewals: {len(ren)} | open: {ren['is_open'].sum()} | closed: {len(ren_closed)}")
print("Open renewals by renewal_number:\n", ren.loc[ren["is_open"], "renewal_number"].value_counts())


def renewal_rates(frame, by):
    return frame.groupby(by)["is_success"].agg(count="count", renewed="sum", rate="mean")


print("\nBy expiry year: all renewals vs closed only")
print(pd.concat({"all": renewal_rates(ren, "expiry_year"),
                 "closed_only": renewal_rates(ren_closed, "expiry_year")}, axis=1))

for by in ["renewal_number", ["renewal_number", "expiry_year"], "acquisition_channel", "plan_code"]:
    print(f"\nRenewal rate by {by} (closed only):")
    print(renewal_rates(ren_closed, by))

Data cut-off (max event time): 2026-10-09 14:00:00
Total renewals: 11580 | open: 600 | closed: 10980
Open renewals by renewal_number:
 renewal_number
1    600
Name: count, dtype: int64

By expiry year: all renewals vs closed only
              all                   closed_only             
            count renewed      rate       count renewed rate
expiry_year                                                 
2024         2500    2000  0.800000        2500    2000  0.8
2025         4000    3200  0.800000        4000    3200  0.8
2026         5080    3824  0.752756        4480    3584  0.8

Renewal rate by renewal_number (closed only):
                count  renewed      rate
renewal_number                          
1                5780     4104  0.710035
2                3400     3060  0.900000
3                1800     1620  0.900000

Renewal rate by ['renewal_number', 'expiry_year'] (closed only):
                            count  renewed  rate
renewal_number expiry_year           

In [10]:
# Why were closed renewals lost: no payment attempt at all, or attempts that failed?
attempts = (
    events[events["event_type"] == "PAYMENT_ATTEMPT"]
    .groupby("renewal_id")
    .agg(attempts=("outcome", "count"),
         failures=("outcome", lambda s: (s == "FAILED").sum()))
)
lost = ren_closed[~ren_closed["is_success"]].merge(attempts, on="renewal_id", how="left")
lost[["attempts", "failures"]] = lost[["attempts", "failures"]].fillna(0).astype(int)
lost["loss_reason"] = np.where(lost["attempts"] == 0, "No payment attempt", "Attempted but failed")
print("Lost renewals:", len(lost))
print(lost["loss_reason"].value_counts())

no_attempt = lost.loc[lost["attempts"] == 0, "renewal_id"]
print("\nEvents logged for renewals with no payment attempt:")
print(events[events["renewal_id"].isin(no_attempt)].groupby(["event_type", "outcome"])["renewal_id"].nunique())

Lost renewals: 2196
loss_reason
No payment attempt      1520
Attempted but failed     676
Name: count, dtype: int64

Events logged for renewals with no payment attempt:
event_type       outcome    
BENEFIT_USED     SUCCESS        1520
REMINDER_SENT    DELIVERED      1358
                 UNDELIVERED     162
SUPPORT_CONTACT  LOGGED          474
Name: renewal_id, dtype: int64


In [11]:
# Event diagnostics
print("Detail codes:\n", events["detail_code"].value_counts())
print("\nPayment-attempt detail codes:\n",
      events.loc[events["event_type"] == "PAYMENT_ATTEMPT", "detail_code"].value_counts())
print("\nFailed-event detail codes:\n",
      events.loc[events["outcome"] == "FAILED", "detail_code"].value_counts())
print("\nReminder delivery outcomes:\n",
      events.loc[events["event_type"] == "REMINDER_SENT", "outcome"].value_counts())
print("\nEvent cost by type:\n", events.groupby("event_type")["event_cost"].agg(["count", "sum", "mean"]))

Detail codes:
 detail_code
FREE_DELIVERY                81803
MEMBER_OFFER                 31588
PRIORITY_SUPPORT             12604
RENEWAL_REMINDER             11580
PSP_TIMEOUT                   2310
AUTHENTICATION_FAILED         1105
INSUFFICIENT_FUNDS             869
FREE_DELIVERY_UNAVAILABLE      620
BENEFITS_QUERY                 364
RENEWAL_HELP                   104
PRICE_QUERY                     86
Name: count, dtype: int64

Payment-attempt detail codes:
 detail_code
PSP_TIMEOUT              2310
AUTHENTICATION_FAILED    1105
INSUFFICIENT_FUNDS        869
Name: count, dtype: int64

Failed-event detail codes:
 detail_code
PSP_TIMEOUT              2310
AUTHENTICATION_FAILED    1105
INSUFFICIENT_FUNDS        869
Name: count, dtype: int64

Reminder delivery outcomes:
 outcome
DELIVERED      10374
UNDELIVERED     1206
Name: count, dtype: int64

Event cost by type:
                   count      sum       mean
event_type                                 
BENEFIT_USED     125995  4526